# Kronos Forecast

Run cells top to bottom. **Cell 2 (model load) only needs to run once per session** — after that you can re-run the forecast cell with different parameters instantly.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = os.getcwd()
sys.path.append(REPO_ROOT)

import torch
print("python  :", sys.version.split()[0])
print("torch   :", torch.__version__)
print("mps     :", torch.backends.mps.is_available())

## 1. Load model and tokenizer

Downloads from Hugging Face on first run (~115 MB), cached afterwards.

`device` is forced to `cpu` because the MPS backend segfaults on some ops. If you want to try the GPU later, change it to `"mps"` — if the kernel dies, switch back.

In [ ]:
from model import Kronos, KronosTokenizer, KronosPredictor

DEVICE = "cpu"

tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
model = Kronos.from_pretrained("NeoQuasar/Kronos-small")
predictor = KronosPredictor(model, tokenizer, device=DEVICE, max_context=512)
print("predictor ready on", DEVICE)

## 2. Load data

Your clone is missing `data/XSHG_5min_600977.csv`, so this uses `tests/data/regression_input.csv` — same schema, 2500 rows of 5-minute bars.

To use your own data, point `CSV_PATH` at any CSV with columns `timestamps, open, high, low, close, volume, amount`.

In [ ]:
CSV_PATH = os.path.join(REPO_ROOT, "tests", "data", "regression_input.csv")

df = pd.read_csv(CSV_PATH)
df["timestamps"] = pd.to_datetime(df["timestamps"])
print(df.shape)
df.head()

## 3. Forecast

`lookback` must stay at or below 512 for Kronos-small/base. Raising `sample_count` averages multiple sampled paths — smoother, but linearly slower on CPU.

In [ ]:
lookback = 400
pred_len = 120

x_df = df.loc[: lookback - 1, ["open", "high", "low", "close", "volume", "amount"]]
x_timestamp = df.loc[: lookback - 1, "timestamps"]
y_timestamp = df.loc[lookback : lookback + pred_len - 1, "timestamps"]

pred_df = predictor.predict(
    df=x_df,
    x_timestamp=x_timestamp,
    y_timestamp=y_timestamp,
    pred_len=pred_len,
    T=1.0,          # sampling temperature
    top_p=0.9,      # nucleus sampling
    sample_count=1, # forecast paths to average
    verbose=True,
)
pred_df.head()

## 4. Plot

In [ ]:
%matplotlib inline

kline_df = df.loc[: lookback + pred_len - 1]
pred_df.index = kline_df.index[-pred_df.shape[0] :]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax1.plot(kline_df["close"], label="Ground Truth", color="blue", linewidth=1.5)
ax1.plot(pred_df["close"], label="Prediction", color="red", linewidth=1.5)
ax1.axvline(lookback - 1, color="gray", linestyle="--", linewidth=1)
ax1.set_ylabel("Close Price", fontsize=13)
ax1.legend(loc="lower left")
ax1.grid(True, alpha=0.3)

ax2.plot(kline_df["volume"], label="Ground Truth", color="blue", linewidth=1.5)
ax2.plot(pred_df["volume"], label="Prediction", color="red", linewidth=1.5)
ax2.axvline(lookback - 1, color="gray", linestyle="--", linewidth=1)
ax2.set_ylabel("Volume", fontsize=13)
ax2.legend(loc="upper left")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()